# iSWAP incoherent error analysis

Incoherent error budget of the resonant exchange gate, following the local relaxation and
pure-dephasing analysis in Appendix L of Li *et al.*,
[PRX **14**, 041050 (2024)](https://doi.org/10.1103/PhysRevX.14.041050).
Sections 1–5 use a model with local $T_1$ and exponential $T_2$; section 7 separately considers flux noise.

**Theory review, 2026-10-08.** The theoretical text has been proofread and corrected. Code cells and
their stored outputs are preserved from the supplied notebook. Section 8 records the derivations,
independent numerical checks, and implementation questions that still require the external module
or raw data. In historical outputs, the column labelled `exact` means the **sum of isolated-channel
errors**, not the exact infidelity of the combined channel.

**Model** (`iswap_incoherent.py`). With $\hbar=1$, project each qubit onto $\{|0\rangle,|1\rangle\}$,
so $a_q=|0\rangle_q\langle1|$ and $n_q=a_q^\dagger a_q$. The resonant exchange Hamiltonian in the
pump rotating frame is

$$H=g(a_1^\dagger a_2+a_2^\dagger a_1),\qquad g=\frac{\pi}{2t_{\rm iSWAP}}.$$

This assumes constant $g$, compensated detuning, independent local Markovian noise, downward
relaxation, and no leakage. The collapse operators are $\sqrt{1/T_1^{(q)}}\,a_q$ and
$\sqrt{2/T_\varphi^{(q)}}\,n_q$, where

$$\frac1{T_\varphi^{(q)}}=\frac1{T_2^{(q)}}-\frac1{2T_1^{(q)}}\ge0.$$

Thus $T_2^{(q)}\le2T_1^{(q)}$ in this exponential model, and $T_\varphi=\infty$ is allowed.
The dephasing operator alone damps a single-qubit coherence as $e^{-t/T_\varphi}$.
For positive $g$, $e^{-iHt_{\rm iSWAP}}$ has exchange amplitudes $-i$; the convention with $+i$
differs by local $Z$ phases. The ideal unitary removed in a fidelity calculation must match the
Hamiltonian and the intended phase corrections.

**Average gate fidelity.** If $A_k$ are the implemented channel's Kraus operators and $U_0$ is
the ideal gate, use $K_k=U_0^\dagger A_k$ in

$$F_{\rm ave}=\frac{1}{d(d+1)}\sum_k\left[\operatorname{Tr}(K_k^\dagger K_k)+|\operatorname{Tr}K_k|^2\right],\qquad d=4.$$

The following finite-time formulas describe an isolated local noise channel embedded in the
two-qubit Hilbert space. For each qubit $q$,

$$p_1^{(q)}=1-e^{-t/T_1^{(q)}},\qquad p_\varphi^{(q)}=\frac{1-e^{-t/T_\varphi^{(q)}}}{2},$$

$$\text{(L8)}\quad r_1^{(q)}=1-\frac{3-p_1^{(q)}+2\sqrt{1-p_1^{(q)}}}{5}
\simeq\frac25\frac{t}{T_1^{(q)}},$$

$$\text{corrected (L13)}\quad r_\varphi^{(q)}=\frac45p_\varphi^{(q)}
\simeq\frac25\frac{t}{T_\varphi^{(q)}}.$$

**Published typo in Eq. (L13).** The published PDF writes $1-F_{\rm ave}^{(q)}=1-4p_\varphi^{(q)}/5$.
Its right-hand side is the fidelity, not the infidelity. In fact,
$K_0=\sqrt{1-p_\varphi}\,I_4$ and $K_1=\sqrt{p_\varphi}\,(Z\otimes I)$ give

$$F_{\rm ave}=\frac{4+16(1-p_\varphi)}{20}=1-\frac45p_\varphi.$$

The corrected form agrees with the paper's small-$t$ expansion and is the form used here.

**Combined budget, to first order.** Additivity holds at first order in the noise rates:

$$r_{\rm lin}\equiv1-F_{\rm ave}\simeq\frac25t\sum_{q=1,2}
\left(\frac1{T_1^{(q)}}+\frac1{T_\varphi^{(q)}}\right)
=\frac25t\sum_q\left(\frac1{T_2^{(q)}}+\frac1{2T_1^{(q)}}\right).$$

The `linear` column is this expression. The historical `exact` column is
$r_{\rm sum}=\sum_q(r_1^{(q)}+r_\varphi^{(q)})$, with the exponential probabilities retained.
Each summand is exact for its isolated channel, but their sum omits composition terms and is
**not** an exact combined-channel or iSWAP result. The full propagator supplies the gate infidelity.
For an independent idle channel the exact expression is given in section 2.
`channel_errors` reports the isolated probabilities and contributions per qubit.

In [ ]:
%matplotlib inline
%load_ext autoreload
%autoreload 2

import sys
import numpy as np
import matplotlib.pyplot as plt


from iswap_incoherent import (
    exchange_curve, exchange_envelope, gate_error, analytic_error, channel_errors,
    exchange_model, fit_exchange, mean_times, t_phi, T_ISWAP, US, NS,
)

# one fixed colour per entity, never cycled
COLORS = ["#2a78d6", "#1baf7a", "#eda100", "#eb6834"]
plt.rcParams.update({
    "figure.dpi": 110,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "grid.linewidth": 0.6,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.titlesize": 10,
    "legend.frameon": False,
    "legend.fontsize": 8,
    "lines.linewidth": 1.6,
})

print("t_iSWAP = %.0f ns, g/2pi = %.4f MHz" % (T_ISWAP / NS, 1 / (4 * T_ISWAP) / 1e6))

## 1. Population exchange curves

One excitation starting in Q1, $|10\rangle$, both qubits with the same $T_1$, $T_2$, over
80 us (about 81 swap periods), sampled every 1 ns. The dense trace is drawn faintly, with its
sampled upper and lower curves on top ($P_{Q1}$ at even and odd multiples of the nominal $t_{\rm iSWAP}$; these are not exact extrema when damping shifts their times); the
zoom below shows the oscillation itself. $T_1$ pulls the whole oscillation down and leaves the minima at zero; pure dephasing lifts
the minima towards the midline.

In [ ]:
T_MAX = 80 * US
DT = 1 * NS                                   # dense time step
t_dense = np.arange(0, T_MAX + DT / 2, DT)
t1_values = np.array([10, 20, 40, 80]) * US
t2_values = np.array([80, 40, 20, 10]) * US

# faint: dense trace; solid / dashed: P(Q1) at even (top) / odd (bottom) multiples of t_iSWAP
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8), sharey=True)
for c, t1 in zip(COLORS, t1_values):
    p1, _ = exchange_curve(t_dense, t1, 2 * t1)
    axes[0].plot(t_dense / US, p1, color=c, lw=0.3, alpha=0.25)
    t_top, top, t_bot, bot = exchange_envelope(T_MAX, t1, 2 * t1)
    axes[0].plot(t_top / US, top, color=c, label="T1 = %.0f us" % (t1 / US))
    axes[0].plot(t_bot / US, bot, color=c, ls="--")
axes[0].set(title="T1 only (T2 = 2 T1)", xlabel="time (us)", ylabel="P(Q1 excited)")

for c, t2 in zip(COLORS, t2_values):
    p1, _ = exchange_curve(t_dense, 40 * US, t2)
    axes[1].plot(t_dense / US, p1, color=c, lw=0.3, alpha=0.25)
    t_top, top, t_bot, bot = exchange_envelope(T_MAX, 40 * US, t2)
    axes[1].plot(t_top / US, top, color=c, label="T2 = %.0f us" % (t2 / US))
    axes[1].plot(t_bot / US, bot, color=c, ls="--")
axes[1].set(title="T1 = 40 us, varying T2", xlabel="time (us)")
for ax in axes:
    ax.legend(loc="upper right", title="solid top, dashed bottom envelope", title_fontsize=8)
fig.tight_layout()

In [ ]:
t_short = np.linspace(0, 4 * T_ISWAP, 801)
fig, ax = plt.subplots(figsize=(6.5, 4))
for c, t2 in zip(COLORS, t2_values):
    p1, p2 = exchange_curve(t_short, 40 * US, t2)
    ax.plot(t_short / NS, p1, color=c, label="T2 = %.0f us" % (t2 / US))
    ax.plot(t_short / NS, p2, color=c, ls="--")
for k in (1, 3):
    ax.axvline(k * T_ISWAP / NS, color="0.5", lw=0.6)
ax.set(title="first swaps at T1 = 40 us (solid Q1, dashed Q2)",
       xlabel="time (ns)", ylabel="population")
ax.legend(loc="center right");

## 2. iSWAP gate error: master equation vs local-channel budget

`gate_error` is intended to build the full $16\times16$ superoperator at $t_{\rm iSWAP}$,
remove the matching ideal unitary, and return $1-F_{\rm ave}$. `idle=True` uses an idle of the same
duration. For a collapse operator $L$ with its rate already included, the first-order contribution is

$$dr=\frac{d\operatorname{Tr}(L^\dagger L)-|\operatorname{Tr}L|^2}{d(d+1)}\,dt.$$

Both traces are invariant under $L\mapsto U_0^\dagger(t)LU_0(t)$, so the first-order budget is
unchanged by the exchange Hamiltonian. For $d=4$, each local relaxation operator contributes
$(2/5)\gamma_1\,dt$, and each local pure-dephasing operator contributes $(2/5)\gamma_\varphi\,dt$.
This establishes the idle/iSWAP agreement at first order; it does not assert exact finite-time equality.
The trace expression also fixes the meaning of the Hilbert–Schmidt norm used in the original text.

For constant independent local rates during an idle, the exact combined fidelity is

$$\boxed{F_{\rm idle}(t)=\frac15+\frac1{20}\prod_{q=1,2}
\left(1+e^{-t/T_1^{(q)}}+2e^{-t/T_2^{(q)}}\right).}$$

The sum of finite isolated-channel infidelities generally differs from this product expression.
During exchange, use the full time-dependent interaction-picture channel or master-equation
propagator rather than the idle expression. All fidelities here refer to the computational
two-qubit channel; leakage would require an extended model and a stated fidelity convention.

In [ ]:
t2_over_t1 = [2.0, 1.5, 1.0, 0.5]           # 2.0 is the T1 limit (no pure dephasing)
t_gate = np.linspace(0, T_ISWAP, 201)

print("%6s %6s %8s | %9s | %10s %10s %10s %10s | %6s"
      % ("T1", "T2", "Tphi", "P2(tsw)", "1-F sim", "1-F idle", "linear", "exact", "sim/lin"))
print("-" * 94)
rows = []
for t1 in t1_values:
    for r in t2_over_t1:
        t2 = r * t1
        tphi = t_phi(t1, t2)
        _, p2 = exchange_curve(t_gate, t1, t2)
        e_sim = gate_error(t1, t2)
        e_idle = gate_error(t1, t2, idle=True)
        e_lin = analytic_error(t1, t2)
        e_exact = analytic_error(t1, t2, exact=True)
        rows.append((t1, t2, e_sim, e_lin))
        print("%6.0f %6.0f %8s | %9.5f | %10.3e %10.3e %10.3e %10.3e | %6.3f" % (
            t1 / US, t2 / US, "inf" if np.isinf(tphi) else "%.1f" % (tphi / US),
            p2[-1], e_sim, e_idle, e_lin, e_exact, e_sim / e_lin))
rows = np.array(rows)

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 4.2))
for c, r in zip(COLORS, t2_over_t1):
    sel = np.isclose(rows[:, 1] / rows[:, 0], r)
    ax.loglog(rows[sel, 0] / US, rows[sel, 2], "o", color=c, label="T2 = %.1f T1" % r)
    ax.loglog(rows[sel, 0] / US, rows[sel, 3], "-", color=c, lw=1)
ax.set(title="iSWAP error at %.0f ns: QuTiP (dots) vs (L8)+(L13) (lines)" % (T_ISWAP / NS),
       xlabel="T1 (us)", ylabel="1 - F_ave")
ax.legend();

## 3. Measured device

Per-qubit values as `(Q1, Q2)`. Echo $T_2$ is used below as an **effective exponential**
coherence time. This is a modeling assumption: Hahn echo filters slow noise differently from an
uninterrupted exchange gate. It neither removes all flux noise nor proves that the remaining
dephasing is white or independent. If the echo decay contains a Gaussian component, extract the
exponential component from a joint fit before converting it into a Markovian $T_\varphi$.
Idle coherence times also need not equal the rates during the pump.

Reference coherence at the idle point (no pump):

| | $T_1$ (us) | $T_{2E}$ (us) | $T_2^*$ (us) |
|---|---|---|---|
| Q1 | 64.4 ± 2.1 | 44.4 ± 1.4 | 40.0 ± 1.0 |
| Q2 | 31.4 ± 0.9 | 34.9 ± 1.0 | 29.9 ± 2.0 |

In [ ]:
T1 = (64.4 * US, 31.4 * US)          # idle T1 of Q1, Q2
T2 = (44.4 * US, 34.9 * US)          # idle echo T2 of Q1, Q2
T2_RAMSEY = (40.0 * US, 29.9 * US)   # idle T2* of Q1, Q2

print("Tphi from echo T2:  Q1 %.1f us, Q2 %.1f us" % tuple(t_phi(a, b) / US for a, b in zip(T1, T2)))
print("pair (harmonic mean): T1 = %.1f us, Tphi = %.1f us\n" % tuple(x / US for x in mean_times(T1, T2)))

print("t = t_iSWAP = %.0f ns\n" % (T_ISWAP / NS))
print("%-3s %9s %9s | %11s %11s | %11s %11s" % (
    "", "p1", "p_phi", "(L8) exact", "(L8) lin", "(L13) exact", "(L13) lin"))
for q, row in enumerate(channel_errors(T1, T2), start=1):
    print("Q%-2d %9.3e %9.3e | %11.3e %11.3e | %11.3e %11.3e" % (
        q, row["p1"], row["p_phi"], row["err_relax"], row["lin_relax"],
        row["err_dephase"], row["lin_dephase"]))

print()
print("exact  1-F = %.3e   (sum of the exact columns)" % analytic_error(T1, T2, exact=True))
print("linear 1-F = %.3e   (sum of the lin columns)" % analytic_error(T1, T2))
print("iSWAP  1-F = %.3e   (QuTiP)" % gate_error(T1, T2))
print("idle   1-F = %.3e   (QuTiP)" % gate_error(T1, T2, idle=True))
print("\nwith T2* instead of echo T2: iSWAP 1-F = %.3e (QuTiP), pair Tphi = %.1f us" % (
    gate_error(T1, T2_RAMSEY), mean_times(T1, T2_RAMSEY)[1] / US))

## 4. Closed-form exchange curve in $T_1$ and $T_\varphi$

Call the qubit prepared in $|1\rangle$ the **donor** (d) and the other the **acceptor** (a); the start
state is $|1_d 0_a\rangle$. In the rest of this section $|10\rangle \equiv |1_d0_a\rangle$ and
$|01\rangle \equiv |0_d1_a\rangle$. (In the measured data of section 5 the donor is Q2.)

### 4.1 Model

With the same two-level, constant-rate assumptions as section 0, in the resonant pump rotating frame the parametric exchange is

$$H = g\,\big(a_d^\dagger a_a + a_a^\dagger a_d\big),\qquad g = \frac{\pi}{2\,t_{\rm iSWAP}},$$

and the Lindblad equation $\dot\rho = -i[H,\rho] + \sum_k \big(L_k\rho L_k^\dagger - \tfrac12\{L_k^\dagger L_k, \rho\}\big)$
has, for each qubit $q \in \{d, a\}$,

$$L_{1,q} = \sqrt{\gamma_1^{(q)}}\;a_q,\qquad L_{\varphi,q} = \sqrt{2\gamma_\varphi^{(q)}}\;n_q,
\qquad \gamma_1^{(q)} = \frac{1}{T_1^{(q)}},\quad \gamma_\varphi^{(q)} = \frac{1}{T_\varphi^{(q)}} = \frac{1}{T_2^{(q)}} - \frac{1}{2T_1^{(q)}} .$$

### 4.2 Only three states take part

$H$ conserves the number of excitations and the $L_{1,q}$ only lower it, so starting from $|10\rangle$ the
state never leaves $\{|10\rangle, |01\rangle, |00\rangle\}$ ($|11\rangle$ is never reached), and no coherence with
$|00\rangle$ is ever created. Write the one-excitation block of $\rho$ as

$$\rho_{dd} = \langle 10|\rho|10\rangle,\qquad \rho_{aa} = \langle 01|\rho|01\rangle,\qquad \rho_{da} = \langle 10|\rho|01\rangle,$$

with $\rho_{00} = 1 - \rho_{dd} - \rho_{aa}$. In this block $H = g\,(|10\rangle\langle 01| + |01\rangle\langle 10|)$.

### 4.3 Equations of motion, term by term

**Exchange.** From $\dot\rho = -i[H,\rho]$, using $(H\rho)_{da} = g\rho_{aa}$, $(\rho H)_{da} = g\rho_{dd}$ and $\rho_{ad} = \rho_{da}^*$:

$$\dot\rho_{dd} = -ig\,(\rho_{ad} - \rho_{da}) = -2g\,\mathrm{Im}\,\rho_{da},\qquad
\dot\rho_{aa} = +2g\,\mathrm{Im}\,\rho_{da},\qquad
\dot\rho_{da} = -ig\,(\rho_{aa} - \rho_{dd}) .$$

The last one is purely imaginary, so $\mathrm{Re}\,\rho_{da}$ stays at its initial value 0.

**Relaxation.** $L_{1,d}$ takes $|10\rangle \to |00\rangle$ and annihilates $|01\rangle$; $L_{1,a}$ does the reverse. The
dissipators give

$$\dot\rho_{dd} = -\gamma_1^{(d)}\rho_{dd},\qquad \dot\rho_{aa} = -\gamma_1^{(a)}\rho_{aa},\qquad
\dot\rho_{da} = -\tfrac12\big(\gamma_1^{(d)} + \gamma_1^{(a)}\big)\rho_{da},$$

and feed only $\rho_{00}$: nothing returns to the one-excitation block.

**Pure dephasing.** $n_d$ is 1 on $|10\rangle$ and 0 on $|01\rangle$, so $L_{\varphi,d}$ leaves the populations alone and damps
the coherence between them at $\tfrac12 \cdot 2\gamma_\varphi^{(d)}\,(1-0)^2 = \gamma_\varphi^{(d)}$; likewise for $L_{\varphi,a}$:

$$\dot\rho_{da} = -\big(\gamma_\varphi^{(d)} + \gamma_\varphi^{(a)}\big)\rho_{da} .$$

### 4.4 Pair rates

Define the **pair rates** as the arithmetic means of the two qubits' rates,

$$\gamma_1 \equiv \tfrac12\big(\gamma_1^{(d)} + \gamma_1^{(a)}\big) \equiv \frac{1}{T_1},\qquad
\gamma_\varphi \equiv \tfrac12\big(\gamma_\varphi^{(d)} + \gamma_\varphi^{(a)}\big) \equiv \frac{1}{T_\varphi},$$

so the pair times $T_1$, $T_\varphi$ are the **harmonic means** of the qubits' times,
$T_1 = 2T_1^{(d)}T_1^{(a)}/(T_1^{(d)} + T_1^{(a)})$ (not the arithmetic mean of the times). `mean_times(t1, t2)`
computes them from per-qubit $T_1$, $T_2$.

### 4.5 Bloch variables

Collect the block into the population left in the pair and a pseudo-spin,

$$P = \rho_{dd} + \rho_{aa},\qquad z = \rho_{dd} - \rho_{aa},\qquad y = 2\,\mathrm{Im}\,\rho_{da},\qquad \Omega = 2g = \frac{\pi}{t_{\rm iSWAP}} .$$

Adding up 4.3, with $\Delta \equiv \tfrac12\big(\gamma_1^{(d)} - \gamma_1^{(a)}\big)$:

$$\begin{aligned}
\dot P &= -\gamma_1 P - \Delta\, z,\\
\dot z &= -\gamma_1 z - \Delta\, P - \Omega\, y,\\
\dot y &= -(\gamma_1 + 2\gamma_\varphi)\, y + \Omega\, z .
\end{aligned}$$

These three equations are exact for unequal relaxation rates within the stated model. In matrix form,

$$\begin{pmatrix}P(t)\\z(t)\\y(t)\end{pmatrix}
=e^{Mt}\begin{pmatrix}1\\1\\0\end{pmatrix},\qquad
M=\begin{pmatrix}
-\gamma_1&-\Delta&0\\
-\Delta&-\gamma_1&-\Omega\\
0&\Omega&-(\gamma_1+2\gamma_\varphi)
\end{pmatrix}.$$

For fast exchange, a useful small parameter is

$$\epsilon_\Delta=\frac{|\Delta|}{\Omega}
=\frac{|1/T_1^{(d)}-1/T_1^{(a)}|\,t_{\rm iSWAP}}{2\pi}.$$

It is $3.26\times10^{-3}$ for 40 and 15 us at 492 ns. Neglecting $\Delta$ produces small
**absolute population corrections**, often of order $\epsilon_\Delta$ over the relevant window;
this is not a uniform relative-error bound near a population minimum or at arbitrarily long times.
The leading fast term has a small period average, but higher-order frequency and decay shifts
can accumulate. In particular, $\dot P(0)=-\gamma_1^{(d)}$, so the harmonic-mean exponential is
not exact at the start when the qubits have different $T_1$ values. Use $e^{Mt}$ when this matters.
Setting $\Delta=0$ (exact for equal $T_1$, approximate otherwise):

$$\dot P = -\gamma_1 P,\qquad \dot z = -\gamma_1 z - \Omega\, y,\qquad \dot y = -(\gamma_1 + 2\gamma_\varphi)\, y + \Omega\, z .$$

### 4.6 Solve

**Population in the pair.** With $P(0) = 1$: $\;P(t) = e^{-\gamma_1 t}$.

**Take out the relaxation.** Put $z = e^{-\gamma_1 t}\,\tilde z$, $y = e^{-\gamma_1 t}\,\tilde y$; the $\gamma_1$ terms cancel:

$$\dot{\tilde z} = -\Omega\,\tilde y,\qquad \dot{\tilde y} = \Omega\,\tilde z - 2\gamma_\varphi\,\tilde y .$$

**Damped oscillator.** Differentiate the first and use the second, then the first again ($\Omega\tilde y = -\dot{\tilde z}$):

$$\ddot{\tilde z} = -\Omega\,\dot{\tilde y} = -\Omega^2\tilde z + 2\gamma_\varphi\,\Omega\tilde y
\quad\Longrightarrow\quad \ddot{\tilde z} + 2\gamma_\varphi\,\dot{\tilde z} + \Omega^2\,\tilde z = 0 .$$

$\tilde z$ behaves as the oscillator position, with velocity $\dot{\tilde z}=-\Omega\tilde y$.
Pure dephasing acts directly on the coherence, not on the populations; exchange converts that
coherence damping into population-oscillation damping. Thus $\tilde y$ is proportional to the
velocity, rather than being the velocity itself.

**Roots.** $s^2 + 2\gamma_\varphi s + \Omega^2 = 0$ gives $s = -\gamma_\varphi \pm i\omega$ with
$\omega = \sqrt{\Omega^2 - \gamma_\varphi^2}$ (underdamped, since $\gamma_\varphi \ll \Omega$), so

$$\tilde z(t) = e^{-\gamma_\varphi t}\,\big(A\cos\omega t + B\sin\omega t\big) .$$

**Initial conditions.** All population starts on the donor, $\tilde z(0) = 1$, so $A = 1$. There is no coherence
yet, $\tilde y(0) = 0$, so the relaxation-factored imbalance starts at rest: $\dot{\tilde z}(0) = -\Omega\,\tilde y(0) = 0$, i.e.
$-\gamma_\varphi A + \omega B = 0$, so $B = \gamma_\varphi/\omega$. This is the $\sin$ term: without it the slope of $\tilde z$ at $t=0$ would be $-\gamma_\varphi$, contradicting
$\dot{\tilde z}(0)=-\Omega\tilde y(0)=0$ and incorrectly making pure dephasing change the initial population imbalance directly:

$$\tilde z(t) = e^{-\gamma_\varphi t}\Big(\cos\omega t + \frac{\gamma_\varphi}{\omega}\sin\omega t\Big) .$$

### 4.7 Result

The donor and acceptor populations are $\rho_{dd,aa} = (P \pm z)/2$:

$$\boxed{P_{\rm d,a}(t) = \tfrac12\,e^{-t/T_1}\Big[1 \pm e^{-t/T_\varphi}\Big(\cos\omega t + \frac{\sin\omega t}{\omega T_\varphi}\Big)\Big],
\qquad \omega = \sqrt{(\pi/t_{\rm iSWAP})^2 - 1/T_\varphi^2}}$$

Here $+$ is the donor and $-$ the acceptor. The formula is exact for $\Delta=0$ and
$\gamma_\varphi<\Omega$ under the stated model. At critical damping replace the bracketed
oscillatory factor by $e^{-\Omega t}(1+\Omega t)$; for $\gamma_\varphi>\Omega$, put
$\kappa=\sqrt{\gamma_\varphi^2-\Omega^2}$ and use
$e^{-\gamma_\varphi t}[\cosh(\kappa t)+(\gamma_\varphi/\kappa)\sinh(\kappa t)]$.

To avoid confusing the Bloch variable $y$ or the pair population $P$ with the readout signal,
write the measured trace for qubit $q$ as

$$m_q(t)=e_{0,q}+C_q P_q(t),\qquad C_q=1-e_{0,q}-e_{1,q},$$

where $e_{0,q}=1-F_{g,q}$ and $e_{1,q}=1-F_{e,q}$. This assumes time-independent readout errors,
correct initial preparation, and no leakage misclassified as $|0\rangle$ or $|1\rangle$.

**Checks.**

* No decoherence: $P_d = \tfrac12(1 + \cos\Omega t) = \cos^2 gt$ — full transfer at $t_{\rm iSWAP} = \pi/2g$.
* Only $T_1$: $P_d = e^{-t/T_1}\cos^2 gt$ — the minima stay at 0 and the whole curve shrinks.
* Only $T_\varphi$, long times: $P_d \to \tfrac12$ — the pair ends up sharing the excitation 50/50.
* For underdamped exchange, write the oscillatory factor as
  $(\Omega/\omega)e^{-\gamma_\varphi t}\cos[\omega t-\arctan(\gamma_\varphi/\omega)]$.
  The population-oscillation amplitude is therefore
  $(\Omega/2\omega)e^{-(\gamma_1+\gamma_\varphi)t}$.
  Pure dephasing directly damps the coherence at $2\gamma_\varphi$ while leaving the
  populations unchanged in its dissipator; the coupled oscillator has decay rate $\gamma_\varphi$.
  For section 3's values, $\gamma_\varphi/\omega\simeq2.15\times10^{-3}$: the phase correction
  is about 2.15 mrad and the population correction is of order $10^{-3}$.
* For equal $T_1$, the closed form and the exact reduced master equation are mathematically
  identical. Independent matrix-exponential evaluation agrees within about $3\times10^{-13}$
  over an 80 us window. The supplied QuTiP outputs show discrepancies up to
  $7.7\times10^{-5}$ for equal qubits; these are numerical/implementation discrepancies,
  not an approximation in the formula. The unequal-$T_1$ example shows a maximum difference
  of about $3.24\times10^{-3}$, consistent with neglecting $\Delta$.

**Reading a fit.** In the $\Delta=0$ model, $T_1$ sets the overall exponential loss,
whereas $T_\varphi$ damps the oscillation contrast and also causes the small frequency/phase
correction above. The reduced fit contains only the two pair rates. This does not prove that an
ideal measurement of the full asymmetric dynamics cannot resolve the relaxation difference:
$\Delta$ affects the exact equations and can in principle be inferred with sufficient accuracy.
For independent local dephasing, the one-excitation block depends only on the sum of its two rates.

Under independent local Markovian noise, the first-order budget needs just these pair rates:

$$r_{\rm lin}=\frac45t_{\rm iSWAP}\left(\frac1{T_1}+\frac1{T_\varphi}\right).$$

The independence assumption is essential when inferring the full gate budget from a population
trace. A common fluctuation couples to $N=n_d+n_a$, which is the identity in the one-excitation
manifold. It is invisible there but dephases superpositions between different excitation sectors.
With correlated noise, the exchange trace measures differential dephasing, not the sum of
the individual dephasing rates. It cannot by itself determine the full two-qubit average fidelity.
The fitted `gate_error` is consequently a conditional model estimate, not a direct measurement
of gate infidelity.

The damped-oscillator form is the resonant transient-nutation (damped Rabi) solution of the Bloch equations
[H. C. Torrey, [Phys. Rev. **76**, 1059 (1949)](https://doi.org/10.1103/PhysRev.76.1059); L. Allen and J. H. Eberly, *Optical Resonance and Two-Level
Atoms* (Wiley, 1975)], here for the one-excitation pseudo-spin, whose population difference has no
relaxation of its own, with the loss out of the manifold as the extra factor $e^{-t/T_1}$.

`exchange_model(t, t1, tphi, iswap_time, e0, e1, acceptor=False)` is this formula;
`fit_exchange(t, y, e0=None, e1=None, acceptor=False)` fits it (in rates, so $T_\varphi = \infty$ is
allowed; pass `e0`, `e1` to hold the readout errors fixed) and returns the implied gate error.

In [ ]:
# formula vs QuTiP, 80 us at 1 ns
t = np.arange(0, 80 * US + 0.5 * NS, 1 * NS)
cases = [((40 * US, 40 * US), (20 * US, 20 * US)),
         ((10 * US, 10 * US), (5 * US, 5 * US)),
         ((80 * US, 80 * US), (160 * US, 160 * US)),
         ((40 * US, 15 * US), (30 * US, 20 * US))]          # unequal qubits

print("%-16s %-16s %-22s %12s" % ("T1 (Q1, Q2) us", "T2 (Q1, Q2) us", "pair T1, Tphi (us)", "max |diff|"))
for t1, t2 in cases:
    p1, _ = exchange_curve(t, t1, t2)
    t1_pair, tphi_pair = mean_times(t1, t2)          # harmonic means of the per-qubit times
    diff = np.abs(exchange_model(t, t1_pair, tphi_pair) - p1).max()
    print("%-16s %-16s %-22s %12.2e" % (tuple(x / US for x in t1), tuple(x / US for x in t2),
                                     "%.1f, %.1f" % (t1_pair / US, tphi_pair / US), diff))

In [ ]:
# fit a simulated measurement: QuTiP trace, readout error, 1000 shots per point
rng = np.random.default_rng(1)
t_meas = np.arange(0, 30 * US, 20 * NS)
T1_true, T2_true = (40 * US, 25 * US), (30 * US, 20 * US)
E0, E1 = 0.01, 0.10                                   # readout: F_g = 0.99, F_e = 0.90

p1, _ = exchange_curve(t_meas, T1_true, T2_true)
y_meas = rng.binomial(1000, E0 + (1 - E0 - E1) * p1) / 1000
result, fit = fit_exchange(t_meas, y_meas)

t1_pair, tphi_pair = mean_times(T1_true, T2_true)     # harmonic means of the per-qubit times
print("                   fitted              true")
print("T1 (pair)       %6.2f +- %.2f us   %6.2f us" % (fit["t1"] / US, fit["t1_err"] / US, t1_pair / US))
print("Tphi (pair)     %6.2f +- %.2f us   %6.2f us" % (fit["tphi"] / US, fit["tphi_err"] / US, tphi_pair / US))
print("t_iSWAP         %6.2f ns           %6.2f ns" % (fit["iswap_time"] / NS, T_ISWAP / NS))
print("e0, e1          %.3f, %.3f        %.3f, %.3f" % (fit["e0"], fit["e1"], E0, E1))
print("\ngate error from the fit, 4/5 t (1/T1 + 1/Tphi): %.3e" % fit["gate_error"])
print("QuTiP gate error with the true per-qubit values:  %.3e" % gate_error(T1_true, T2_true))

fig, axes = plt.subplots(1, 2, figsize=(11, 3.6), gridspec_kw={"width_ratios": [2, 1]})
t_fine = np.linspace(0, t_meas[-1], 30001)
for ax, xlim in zip(axes, [(0, 30), (0, 3)]):
    ax.plot(t_meas / US, y_meas, ".", color=COLORS[0], ms=2, label="simulated data")
    ax.plot(t_fine / US, exchange_model(t_fine, fit["t1"], fit["tphi"], fit["iswap_time"],
                                        fit["e0"], fit["e1"]),
            color=COLORS[3], lw=0.8, label="fit")
    ax.set(xlim=xlim, xlabel="time (us)")
axes[0].set(ylabel="P(donor read as 1)", title="exchange_model fit to a simulated trace")
axes[1].set(title="first 3 us")
axes[0].legend(loc="upper right")
fig.tight_layout()

## 5. Measured trace: 20260924-204141 (SNAIL iSWAP)

Continuous pump at 9 dBm, 258.667 MHz; pump duration 0 – 50 us in 60 ns steps, 100 shots per point,
thresholded readout (`SNL315.ipynb`, section 5.2). **Q2 is the donor** (`qubit_e = [qubit2]`) and Q1 the
acceptor; `y1` is the Q2 population and `y0` the Q1 population (as in `iswap_chevron_fit`).
Single-shot readout fidelities $F_g \approx 0.99$, $F_e \approx 0.90$ give $e_0 \approx 0.01$, $e_1 \approx 0.10$.

Both traces are fitted with section 4's reduced model, once with the readout errors free and once
held at the calibration. Under that model, $T_1$ and $T_\varphi$ estimate harmonic means during
the fitted pump window. They are effective values if the rates vary with time, the readout changes
when the pump turns on, or the dephasing is correlated.

The printed `redchi` values are compared with a binomial variance on the probability-squared scale.
For an unweighted fit, this quantity is a residual mean square, not a statistically normalized
reduced $\chi^2$. A binomial-weighted reduced $\chi^2$ has expectation approximately 1.
The external `fit_exchange` implementation is needed to confirm its weighting convention.

In [ ]:
import os
from quantify_core.data.handling import set_datadir, load_dataset
DATA_DIR = os.environ.get("QBLOX_DATADIR", "../CD1")  # quantify data dir; override with QBLOX_DATADIR
set_datadir(DATA_DIR)

TUID = "20260924-204141-727-2894c5"
E0_CAL, E1_CAL = 0.01, 0.10
ds_meas = load_dataset(TUID)
t_data = ds_meas.x0.values
traces = {"donor Q2": (ds_meas.y1.values, False), "acceptor Q1": (ds_meas.y0.values, True)}

fits = {}
print("%-34s %12s %13s %9s %6s %6s %9s %9s" % (
    "", "T1 (us)", "Tphi (us)", "t_sw (ns)", "e0", "e1", "red. chi2", "1-F"))
for name, (y, acceptor) in traces.items():
    for ro, kw in (("readout free", {}), ("readout fixed", dict(e0=E0_CAL, e1=E1_CAL))):
        result, f = fit_exchange(t_data, y, acceptor=acceptor, **kw)
        fits[name, ro] = (result, f)
        print("%-34s %6.1f+-%-4.1f %6.1f+-%-5.1f %9.2f %6.3f %6.3f %9.2e %9.3e" % (
            name + ", " + ro, f["t1"] / US, f["t1_err"] / US, f["tphi"] / US, f["tphi_err"] / US,
            f["iswap_time"] / NS, f["e0"], f["e1"], result.redchi, f["gate_error"]))

# 100 shots per point: binomial variance p(1-p)/100 sets the reduced chi^2 a perfect model reaches
p = fits["donor Q2", "readout free"][0].best_fit
print("\nshot-noise floor for the reduced chi2: %.2e" % np.mean(p * (1 - p) / 100))

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(11, 8.5), gridspec_kw={"height_ratios": [2, 1, 1.4]})
t_fine = np.linspace(0, t_data[-1], 50001)
for c, (name, (y, acceptor)) in zip((COLORS[0], COLORS[3]), traces.items()):
    _, f = fits[name, "readout free"]
    curve = exchange_model(t_fine, f["t1"], f["tphi"], f["iswap_time"], f["e0"], f["e1"], acceptor=acceptor)
    for ax in (axes[0], axes[2]):
        ax.plot(t_data / US, y, ".", color=c, ms=2.5, label=name)
        ax.plot(t_fine / US, curve, color=c, lw=0.7)
    res = y - fits[name, "readout free"][0].best_fit
    axes[1].plot(t_data / US, res, ".", color=c, ms=2, alpha=0.5)
    bins = np.array_split(np.arange(t_data.size), 20)
    axes[1].plot([t_data[b].mean() / US for b in bins], [res[b].mean() for b in bins], "-o",
                 color=c, ms=3, lw=1.2)
axes[0].set(ylabel="P(read as 1)", title="%s: data and fit (readout free)" % TUID)
axes[0].legend(loc="upper right")
axes[1].axhline(0, color="0.5", lw=0.6)
axes[1].set(ylabel="residual", title="residuals (dots) and 2.5 us bin means (lines)")
axes[2].set(xlim=(0, 5), xlabel="pump duration (us)", ylabel="P(read as 1)", title="first 5 us")
for ax in axes[:2]:
    ax.set_xlim(0, t_data[-1] / US)
fig.tight_layout()

### Is the excitation loss a single exponential?

The binned residuals of both traces share a slow pattern, suggesting limitations of the
constant-rate fit. Summing **calibrated** populations gives the pair excitation population:

$$P(t)=\frac{m_d(t)-e_{0,d}}{C_d}+\frac{m_a(t)-e_{0,a}}{C_a}.$$

The raw sum is instead

$$m_d+m_a=e_{0,d}+e_{0,a}+\frac{C_d+C_a}{2}P+\frac{C_d-C_a}{2}z.$$

It cancels the oscillation only if the readout contrasts are equal and stable. Even for equal
contrasts, $P=e^{-t/T_1}$ is exact only for equal relaxation rates in the constant-rate model;
unequal rates produce the small corrections in section 4.5. The comparisons below use the raw sum
as an approximate diagnostic, with a single exponential $c+a e^{-t/T}$, a stretched exponential
$c+a e^{-(t/T)^n}$, and local log slopes. The local slopes in the code do not subtract the
readout offset $c$, so their decay times are effective diagnostics rather than direct $T_1$ values.

In [ ]:
import lmfit

total = ds_meas.y0.values + ds_meas.y1.values

def single_exp(t, a, tau, c):
    return c + a * np.exp(-t / tau)

def stretched_exp(t, a, tau, c, n):
    return c + a * np.exp(-(t / tau) ** n)

fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
axes[0].plot(t_data / US, total, ".", color="0.6", ms=2, label="Q1 + Q2")
for c, (label, func, guess) in zip((COLORS[0], COLORS[3]), (
        ("single exp", single_exp, dict(a=0.9, tau=25 * US, c=0.02)),
        ("stretched exp", stretched_exp, dict(a=0.9, tau=25 * US, c=0.02, n=1.0)))):
    model = lmfit.Model(func)
    params = model.make_params(**guess)
    params["c"].set(min=0)
    r = model.fit(total, params, t=t_data)
    n = r.params["n"] if "n" in r.params else None
    print("%-14s tau = %.1f us%s   BIC = %.0f" % (
        label, r.params["tau"].value / US,
        "" if n is None else ", n = %.2f +- %.2f" % (n.value, n.stderr), r.bic))
    axes[0].plot(t_data / US, r.best_fit, color=c, lw=1.2, label=label)

print("\nlocal decay time of Q1 + Q2:")
starts = np.arange(0, 50, 10)
local = []
for a in starts:
    m = (t_data >= a * US) & (t_data < (a + 10) * US)
    local.append(-1 / np.polyfit(t_data[m], np.log(np.clip(total[m], 1e-3, None)), 1)[0])
    print("  %2d - %2d us: %.1f us" % (a, a + 10, local[-1] / US))

axes[0].set(xlabel="pump duration (us)", ylabel="Q1 + Q2", title="population left in the pair")
axes[0].legend()
axes[1].bar(starts + 5, np.array(local) / US, width=8, color=COLORS[0])
axes[1].set(xlabel="window centre (us)", ylabel="local decay time (us)",
            title="decay time in 10 us windows")
fig.tight_layout()

The stretched-exponential fit gives $n>1$, and the raw-sum log slope becomes steeper with pump
duration. These observations are consistent with a time-dependent loss rate, but readout drift,
preparation changes, unequal contrasts, or model mismatch can also contribute. A single fitted
$T_1$ therefore depends on the window. The gate lasts 492 ns, so its first-order budget depends on
rates integrated over that interval:

$$r_{\rm lin}\simeq\frac25\int_0^{t_{\rm iSWAP}}dt\sum_q
[\gamma_1^{(q)}(t)+\gamma_\varphi^{(q)}(t)].$$

Growing-window fits are useful checks, but 10–20 us effective rates do not establish the first
492 ns rate, especially with an apparent turn-on step. Extrapolating the fitted stretched
exponential to $t=0$ is also not a physical measurement of the initial loss rate.

In [ ]:
print("%-8s %-14s %12s %13s %6s %6s %9s" % ("window", "", "T1 (us)", "Tphi (us)", "e0", "e1", "1-F"))
for t_max in (10, 15, 20, 50):
    m = t_data <= t_max * US
    for ro, kw in (("readout free", {}), ("readout fixed", dict(e0=E0_CAL, e1=E1_CAL))):
        _, f = fit_exchange(t_data[m], ds_meas.y1.values[m], **kw)
        print("0-%-5s %-14s %6.1f+-%-4.1f %6.1f+-%-5.1f %6.3f %6.3f %9.3e" % (
            "%d us" % t_max, ro, f["t1"] / US, f["t1_err"] / US, f["tphi"] / US, f["tphi_err"] / US,
            f["e0"], f["e1"], f["gate_error"]))

### Step in the total population when the pump turns on

With the readout fixed, $T_1$ drops to 21 – 24 us; with it free, $e_1$ climbs to 0.17 in the short
windows. Both point at the start of the trace: Q1 + Q2 at $t = 0$ (no pump) is compared with the first
microseconds of pump and with the 2 – 10 us exponential extrapolated back to $t = 0$.

In [ ]:
print("Q1 + Q2, bin means (100 shots per point per qubit):")
for a, b in [(0, 0.06), (0.06, 0.5), (0.5, 1), (1, 2), (2, 3), (3, 5), (5, 8)]:
    m = (t_data >= a * US) & (t_data < b * US)
    err = total[m].std() / np.sqrt(m.sum()) if m.sum() > 1 else np.sqrt(2 * 0.92 * 0.08 / 100)
    print("  %4.2f - %4.2f us  (%2d pts)  %.3f +- %.3f" % (a, b, m.sum(), total[m].mean(), err))

m = (t_data >= 2 * US) & (t_data < 10 * US)
slope, intercept = np.polyfit(t_data[m], np.log(total[m]), 1)
print("\n2 - 10 us: exponential with tau = %.1f us, extrapolated to t = 0: %.3f" % (
    -1 / slope / US, np.exp(intercept)))

### Idle coherence vs the pumped swap

Section 3's idle values (no pump) predict the pair values $T_1 = 42.2$ us and $T_\varphi = 72.8$ us
(echo). Draw the curve they predict — with the fitted $t_{\rm iSWAP}$ and the calibrated readout —
over the data, next to the fit.

In [ ]:
t1_idle, tphi_idle = mean_times(T1, T2)             # section 3: idle T1, echo T2
t1_idle_r, tphi_idle_r = mean_times(T1, T2_RAMSEY)   # idle T1, T2*
_, f_free = fits["donor Q2", "readout free"]
_, f_early = fit_exchange(t_data[t_data <= 20 * US], ds_meas.y1.values[t_data <= 20 * US])

print("%-38s %9s %10s %9s" % ("", "T1 (us)", "Tphi (us)", "1-F"))
for label, t1_, tphi_ in (("idle, echo T2", t1_idle, tphi_idle),
                          ("idle, T2*", t1_idle_r, tphi_idle_r),
                          ("swap fit, 0-20 us, readout free", f_early["t1"], f_early["tphi"]),
                          ("swap fit, 0-50 us, readout free", f_free["t1"], f_free["tphi"])):
    print("%-38s %9.1f %10.1f %9.3e" % (label, t1_ / US, tphi_ / US,
                                        0.8 * T_ISWAP * (1 / t1_ + 1 / tphi_)))

fig, axes = plt.subplots(2, 1, figsize=(11, 6.5))
t_fine = np.linspace(0, t_data[-1], 50001)
idle_curve = exchange_model(t_fine, t1_idle, tphi_idle, f_free["iswap_time"], E0_CAL, E1_CAL)
fit_curve = exchange_model(t_fine, f_free["t1"], f_free["tphi"], f_free["iswap_time"],
                           f_free["e0"], f_free["e1"])
for ax, xlim in zip(axes, [(0, 50), (0, 5)]):
    ax.plot(t_data / US, ds_meas.y1.values, ".", color="0.55", ms=2.5, label="donor Q2 data")
    ax.plot(t_fine / US, idle_curve, color=COLORS[0], lw=0.8,
            label="prediction from idle T1, T2E (T1 = %.1f, Tphi = %.1f us)" % (t1_idle / US, tphi_idle / US))
    ax.plot(t_fine / US, fit_curve, color=COLORS[3], lw=0.8,
            label="fit (T1 = %.1f, Tphi = %.1f us)" % (f_free["t1"] / US, f_free["tphi"] / US))
    ax.set(xlim=xlim, ylabel="P(Q2 read as 1)")
# midline (P_tot / 2 through the readout) makes the T1 difference visible over the full window
for t1_, c in ((t1_idle, COLORS[0]), (f_free["t1"], COLORS[3])):
    axes[0].plot(t_fine / US, E0_CAL + (1 - E0_CAL - E1_CAL) * 0.5 * np.exp(-t_fine / t1_),
                 color=c, lw=1.6, ls="--")
axes[0].set(title="%s: idle-coherence prediction vs data (dashed: midlines)" % TUID)
axes[0].legend(loc="upper right")
axes[1].set(xlabel="pump duration (us)", title="first 5 us")
fig.tight_layout()

### Summary for 20260924-204141

* **Fit residuals are close to, but above, the shot-noise scale.** The reported residual mean
  squares are about $1.4$–$1.5\times10^{-3}$, compared with a binomial variance estimate of
  $1.2\times10^{-3}$. These numbers are not normalized reduced $\chi^2$ values unless the fit
  weights are verified. The structured residuals show a limitation of the single-rate model.
  Donor and acceptor fits give consistent $t_{\rm iSWAP}\simeq491.9$ ns.
* **Conditional pump-window estimates:** $T_1\simeq22$–$30$ us and
  $T_\varphi\simeq60$–$75$ us, depending on readout assumptions and the fit window.
  Under independent local exponential noise, these imply a first-order budget of roughly
  $1.9$–$2.5\times10^{-2}$ at 492 ns, dominated by relaxation. This is not a directly measured
  average gate infidelity and does not include hidden common dephasing or leakage.
* **Idle comparison:** the supplied idle values give pair $T_1=42.2\pm0.9$ us and
  $T_\varphi=72.8\pm3.2$ us when echo is treated as exponential. The linear budget is
  $1.473\times10^{-2}$; the constant-rate exchange calculation gives $1.458\times10^{-2}$.
  The fitted pump-window $T_\varphi$ values overlap the idle estimate, while the fitted loss
  times are shorter. This supports an increased effective loss contribution under the model;
  it does not exclude pump-induced common noise or determine a sub-microsecond gate budget.
* **The raw excitation signal is not well described by one exponential.** Its local slope
  corresponds to about 30 us in the first 20 us and 19–21 us later; the stretched fit gives
  $n=1.33$. These are window-dependent signal diagnostics. Calibrated populations and a
  model for pump-dependent readout are needed to identify the physical loss rate.
* **An apparent pump-on signal step needs separate calibration.** The raw Q1+Q2 signal
  changes from 0.92 at the single $t=0$ point to about 0.81 at 60–500 ns.
  Population loss, leakage, preparation effects, or a pump-dependent readout contrast could
  contribute; the trace alone does not distinguish them. The zero-duration reference has
  only 100 shots. A denser, higher-shot early-time scan and pump-on readout controls would
  test the step and make a gate-duration error estimate more reliable.

## 6. Pump-power dependence

Moved to [`iswap_power_sweep.ipynb`](iswap_power_sweep.ipynb) (power sweeps of 2026-09-29, per-trace fits, interleaved references, high-power checks, and Q2 $T_1$ with the pump detuned).

## 7. Flux-noise-induced error (Appendix L1)

Li *et al.* refit Hahn-echo decays using
$P=A+B e^{-\Gamma_{\rm exp}t-(\Gamma_\Phi^E t)^2}$ and relate the Gaussian rate to the
flux sensitivity. Writing $s_q=\partial\omega_q/\partial\Phi$ for a derivative with respect to
physical flux, their Eq. (L3) becomes

$$\Gamma_{\Phi,q}^E=\sqrt{A_\Phi\ln2}\,|s_q|.$$

The paper uses a phase coordinate $\varphi_{\rm ex}=2\pi\Phi/\Phi_0$, which accounts for its
explicit conversion factor $2\pi/\Phi_0$. In this notebook the spline coordinate is
$x=\Phi/\Phi_0$, so use $\partial\omega/\partial x=2\pi\,\partial f/\partial x$ and express
$\sqrt{A_x}=\sqrt{A_\Phi}/\Phi_0$ in units of $\mu\Phi_0/\Phi_0$. The code converts $f$
in Hz to angular frequency once. The paper reports $\sqrt{A_\Phi}\simeq4.84\,\mu\Phi_0$.
Its negligible CZ estimate is device- and pulse-specific; it is not an iSWAP error bound.

**Spectral convention.** To make the echo formula and the offset variance unambiguous, define

$$\langle\delta\Phi(t)\delta\Phi(0)\rangle=
\int_{-\infty}^{\infty}d\omega\,\mathcal S_\Phi(\omega)e^{-i\omega t},
\qquad\mathcal S_\Phi(\omega)=\frac{A_\Phi}{|\omega|}.$$

If instead the Fourier integral contains $d\omega/(2\pi)$, its two-sided PSD is
$2\pi A_\Phi/|\omega|$. In the convention above, a band of quasi-static offsets has variance

$$\sigma_\Phi^2=2A_\Phi\ln(\omega_{\rm hi}/\omega_{\rm lo}).$$

One cutoff choice consistent with the stored results is
$\omega_{\rm lo}=2\pi/t_{\rm meas}$ and $\omega_{\rm hi}=1/t_{\rm iSWAP}$.
The exact cutoff and recalibration assumptions must be checked in `flux_noise_iswap_error`;
they are not specified by an echo rate alone. The resulting offset model is an approximation
to the gate filter, not a complete treatment of all $1/f$ frequencies.

The measurements used here are the T1/echo scan of `SNL315_plot.ipynb` section 3.2
(`CD1/multi_t1t2_curr_0809_tuid_arr.txt`, 741 runs, 2026-08-07–08-09: 13 currents
from 0 to 0.75 mA, 57 passes, 10 repeats per pass, and 60 delays to 196 us).

* **Flux sensitivity:** snapshots supply $f_{01}$ at each current. With
  $x=(I-0.0055\,{\rm mA})/(2\times0.8711\,{\rm mA})$, a cubic spline estimates
  $\partial f/\partial x$. At 0.715 mA the stored derivatives are about
  $-63$ and $-107$ MHz per unit $\Phi/\Phi_0$. Derivative uncertainty is not propagated
  in the present noise bounds.
* **Echo fits:** `echo_gaussian_profile` is intended to profile the Gaussian rate with exponential
  decay and readout parameters as nuisance parameters. The rising traces imply $B<0$.
  The nominal upper limits are conditional on the fit and error model; raw data and the
  function implementation are needed to validate their coverage. Several Q2 fit statistics
  substantially exceed 1, including about 4.8 at the selected 0.73 mA point.
  Choosing the smallest of multiple pointwise 95% limits does not automatically produce a
  global 95% limit. A joint fit for a common $A_\Phi$ or simultaneous limits is preferable.
* **Exponential rate:** the code also fits each echo to a *pure* exponential and stores
  $1/T_{2E}^{\rm eff}-1/(2T_1)$. This is an effective single-exponential rate. It is not the
  joint fit's $\Gamma_{\rm exp}-1/(2T_1)$ when a Gaussian component is present, especially
  for Q1. Extract the nuisance exponential rate from the joint fit before interpreting it
  as white dephasing or adding it to a separate Gaussian budget.
* **Shared flux and gate sensitivity:** a common flux offset produces correlated shifts
  $\delta\omega_q=s_q\delta\Phi$. The fixed-$g$ iSWAP sees both common phase noise and
  differential detuning noise; the population exchange sees only the latter.
  Section 8 derives the gate's weak-noise expression. The calculation using only
  `dfdphi_715` does not include $\partial g/\partial\Phi$, pump amplitude/phase noise,
  or other coupler-dependent terms.

In [ ]:
import os
from scipy.interpolate import CubicSpline
from scipy.optimize import curve_fit
from iswap_incoherent import echo_gaussian_profile, flux_noise_iswap_error

FLUX_CACHE = "flux_noise_0809_traces.npz"
if not os.path.exists(FLUX_CACHE):
    from quantify_core.data.handling import load_snapshot
    tuids = np.genfromtxt(os.path.join(DATA_DIR, "multi_t1t2_curr_0809_tuid_arr.txt"), dtype="str")
    rec = {k: [] for k in ("current_A", "f01_q1", "f01_q2", "echo_q1", "echo_q2", "t1_q1", "t1_q2")}
    for tu in tuids:
        snap, ds = load_snapshot(tu)["instruments"], load_dataset(tu)
        rec["current_A"].append(snap["GS210_1"]["parameters"]["current"]["value"])
        for q, ch in (("q1", "y0"), ("q2", "y1")):
            rec["f01_" + q].append(snap["qubit%s" % q[1]]["submodules"]["clock_freqs"]["parameters"]["f01"]["value"])
            n_tau = np.unique(ds.x0.values).size
            rec["echo_" + q].append(ds.where(ds.x1 == 2, drop=True)[ch].values.reshape(-1, n_tau))
            rec["t1_" + q].append(ds.where(ds.x1 == 1, drop=True)[ch].values.reshape(-1, n_tau))
    np.savez_compressed(FLUX_CACHE, tau_s=np.unique(ds.x0.values), **{k: np.array(v) for k, v in rec.items()})
fl = np.load(FLUX_CACHE)
cur_ma = fl["current_A"] * 1e3
tau_us = fl["tau_s"] * 1e6
I_SET = np.unique(np.round(cur_ma, 4))
phi_set = (I_SET - 0.0055) / (2 * 0.8711)                      # Phi / Phi0, as curr2flux in SNL315_plot.ipynb
LN2 = np.log(2)

flux = {}
for q in ("q1", "q2"):
    f01 = np.array([fl["f01_" + q][np.isclose(cur_ma, c)][0] for c in I_SET])
    dfdphi = CubicSpline(phi_set, f01)(phi_set, 1)              # Hz per Phi0
    sens = 2 * np.pi * np.abs(dfdphi) * 1e-6                    # |d omega / d Phi| in 1/s per uPhi0
    rows = []
    for c in I_SET:
        idx = np.isclose(cur_ma, c)
        per_pass = fl["echo_" + q][idx].mean(1)
        y, sig = per_pass.mean(0), per_pass.std(0, ddof=1) / np.sqrt(idx.sum())
        best, up, red = echo_gaussian_profile(tau_us, y, sig, np.linspace(0, 0.03, 121))   # 1/us
        p_exp, _ = curve_fit(lambda t, a, b, g: a + b * np.exp(-g * t), tau_us, y, p0=(0.47, -0.45, 0.02), sigma=sig)
        p_t1, _ = curve_fit(lambda t, a, b, g: a + b * np.exp(-g * t), tau_us, fl["t1_" + q][idx].mean((0, 1)),
                            p0=(0.05, 0.9, 0.02))
        rows.append((best * 1e6, up * 1e6, red, 1 / p_t1[2], 1 / p_exp[2], (p_exp[2] - p_t1[2] / 2) * 1e6))
    rows = np.array(rows)
    flux[q] = dict(f01=f01, dfdphi=dfdphi, sens=sens, g_gauss=rows[:, 0], g_gauss_up=rows[:, 1], redchi=rows[:, 2],
                   t1=rows[:, 3], t2e=rows[:, 4], g_phi_exp=rows[:, 5])

for q in ("q1", "q2"):
    r = flux[q]
    print("%s %6s %8s | %9s %11s | %18s %6s | %6s %6s | %12s | %s" % (
        q.upper(), "I (mA)", "Phi/Phi0", "f01 (GHz)", "df/dPhi", "Gamma_Phi^E (1/ms)", "redchi", "T1", "T2E",
        "exp. deph.", "sqrt(A) bound"))
    print("%3s %6s %8s | %9s %11s | %18s %6s | %6s %6s | %12s | %s" % (
        "", "", "", "", "(MHz/Phi0)", "best   95% upper", "", "(us)", "(us)", "(1/ms)", "(uPhi0, 95 %)"))
    for k, c in enumerate(I_SET):
        bound = r["g_gauss_up"][k] / (np.sqrt(LN2) * r["sens"][k] * 1e6) * 1e6 if r["sens"][k] > 50 else np.nan
        print("    %6.3f %8.4f | %9.6f %11.1f | %7.2f   %7.2f   %6.2f | %6.1f %6.1f | %12.2f | %s" % (
            c, phi_set[k], r["f01"][k] / 1e9, r["dfdphi"][k] / 1e6, r["g_gauss"][k] / 1e3, r["g_gauss_up"][k] / 1e3,
            r["redchi"][k], r["t1"][k], r["t2e"][k], r["g_phi_exp"][k] / 1e3, "-" if np.isnan(bound) else "%.2f" % bound))
    print()

# 1/f amplitude: Q2 has no Gaussian component at any current, so only an upper bound; take the tightest one
b2 = flux["q2"]["g_gauss_up"] / (np.sqrt(LN2) * flux["q2"]["sens"] * 1e6) * 1e6
k_b = np.nanargmin(np.where(flux["q2"]["sens"] > 50, b2, np.nan))
SQRT_A_BOUND = b2[k_b] * 1e-6                                     # Phi0
print("tightest bound, Q2 at %.3f mA: sqrt(A_Phi) <= %.2f uPhi0 (95 %%)   [Li et al.: 4.84 uPhi0]" % (I_SET[k_b], SQRT_A_BOUND * 1e6))

In [ ]:
K715 = int(np.argmin(np.abs(I_SET - 0.715)))
fig, axes = plt.subplots(1, 3, figsize=(14, 3.9))
ax = axes[0]
for q, c in (("q1", COLORS[3]), ("q2", COLORS[0])):
    r = flux[q]
    ax.errorbar(r["sens"], r["g_gauss"] / 1e3, yerr=[np.zeros_like(r["g_gauss"]), (r["g_gauss_up"] - r["g_gauss"]) / 1e3],
                fmt="o", color=c, ms=4, capsize=2, label="%s: best and 95 %% upper" % q.upper())
s_line = np.linspace(0, flux["q2"]["sens"].max(), 50)
ax.plot(s_line, np.sqrt(LN2) * 4.84 * s_line / 1e3, "k--", lw=1, label="Li et al., sqrt(A) = 4.84 uPhi0")
ax.plot(s_line, np.sqrt(LN2) * SQRT_A_BOUND * 1e6 * s_line / 1e3, ":", color=COLORS[0], lw=1.2,
        label="bound here, %.1f uPhi0" % (SQRT_A_BOUND * 1e6))
ax.set(xlabel="|d omega / d Phi| (1/s per uPhi0)", ylabel="Gamma_Phi^E (1/ms)", ylim=(0, 25),
       title="Gaussian (1/f) echo rate vs flux sensitivity")
ax.legend(fontsize=7)

ax = axes[1]
for q, c in (("q1", COLORS[3]), ("q2", COLORS[0])):
    r = flux[q]
    ax.plot(r["sens"], r["g_phi_exp"] / 1e3, "o", color=c, ms=4, label=q.upper())
    pq, _ = curve_fit(lambda x, a, cc: a + cc * x**2, r["sens"], r["g_phi_exp"], p0=(3e3, 2e-2))
    ax.plot(s_line, (pq[0] + pq[1] * s_line**2) / 1e3, "-", color=c, lw=1, label="%s: a + c s^2" % q.upper())
    flux[q]["quad"] = pq
ax.set(xlabel="|d omega / d Phi| (1/s per uPhi0)", ylabel="1/T2E - 1/(2 T1), exponential (1/ms)", yscale="log",
       title="exponential pure dephasing vs flux sensitivity")
ax.legend(fontsize=7)

ax = axes[2]
for q, c in (("q1", COLORS[3]), ("q2", COLORS[0])):
    ax.plot(phi_set, (flux[q]["f01"] - flux[q]["f01"][0]) / 1e6, "o-", color=c, ms=4, lw=1, label=q.upper())
ax.axvline(phi_set[K715], color="0.5", lw=0.7)
ax.set(xlabel="SNAIL flux (Phi/Phi0)", ylabel="f01 - f01(0 mA) (MHz)", title="qubit frequency vs SNAIL flux (0.715 mA marked)")
ax.legend()
fig.tight_layout()

In [ ]:
# (1) the Appendix L1 term for the iSWAP: quasi-static 1/f offsets of the shared SNAIL flux, Monte Carlo on the
#     full 4x4 iSWAP unitary at 0.715 mA, for the bound found here and for the paper's amplitude
dfdphi_715 = (flux["q1"]["dfdphi"][K715], flux["q2"]["dfdphi"][K715])
print("iSWAP at 0.715 mA, t_iSWAP = %.0f ns; df/dPhi = %.1f, %.1f MHz/Phi0 (Q1, Q2)" % (T_ISWAP / NS, *(np.array(dfdphi_715) / 1e6)))
print("%-28s %10s | %18s %18s" % ("1/f amplitude", "", "1-F, t_meas = 1 s", "t_meas = 1 h"))
for label, sa in (("bound here", SQRT_A_BOUND), ("Li et al.", 4.84e-6)):
    e1, _ = flux_noise_iswap_error(sa, dfdphi_715, T_ISWAP, t_meas=1.0)
    e2, s2 = flux_noise_iswap_error(sa, dfdphi_715, T_ISWAP, t_meas=3600.0)
    print("%-17s %5.2f uPhi0 %10s | %18.1e %18.1e   (rms offset %.0f uPhi0)" % (label, sa * 1e6, "", e1, e2, s2 * 1e6))

# (2) the flux-dependent part of the white (exponential) dephasing: already inside the measured T_phi and the (L13)
#     term, attributed here as the rise of the exponential pure-dephasing rate from 0 mA to 0.715 mA
d_gamma = {q: flux[q]["g_phi_exp"][K715] - flux[q]["g_phi_exp"][0] for q in ("q1", "q2")}
tot_gamma = {q: flux[q]["g_phi_exp"][K715] for q in ("q1", "q2")}
err_flux_white = 0.4 * T_ISWAP * sum(d_gamma.values())
err_deph_all = 0.4 * T_ISWAP * sum(tot_gamma.values())
print("\nexponential pure dephasing at 0.715 mA (Aug 2026 data): Q1 %.1f /ms, Q2 %.1f /ms; rise from 0 mA: %.1f, %.1f /ms" % (
    tot_gamma["q1"] / 1e3, tot_gamma["q2"] / 1e3, d_gamma["q1"] / 1e3, d_gamma["q2"] / 1e3))
print("(L13) dephasing error at %.0f ns: %.2e in total, of which %.2e (%.0f %%) is flux-dependent" % (
    T_ISWAP / NS, err_deph_all, err_flux_white, 100 * err_flux_white / err_deph_all))
c2 = flux["q2"]["quad"][1]                                        # s * uPhi0^2:  Gamma = c s^2 = S_Phi s^2 / 2
print("Q2 quadratic fit: Gamma = a + c s^2 with c = %.2e s uPhi0^2 -> white flux noise S_Phi ~ 2c = %.2f uPhi0^2/Hz "
      "(%.2f uPhi0/sqrt(Hz), i.e. %.2f nA/sqrt(Hz) of bias current at 1.742 mA/Phi0)" % (
          c2, 2 * c2, np.sqrt(2 * c2), np.sqrt(2 * c2) * 1e-6 * 1.7422e6))

* **No Gaussian component is resolved on Q2 within this fit model.** The selected pointwise
  limit is $\sqrt{A_\Phi}\lesssim2.3\,\mu\Phi_0$ at 0.73 mA. Its stated 95% interpretation
  needs the fit-quality, selection, and sensitivity-uncertainty qualifications in section 7.
  Q1 has a 5–8 /ms Gaussian component even near zero sensitivity, consistent with another
  slow noise source; that baseline does not exclude an additional flux contribution.
* **The fixed-$g$ frequency-shift contribution is small in the specified offset model:**
  about $(0.4$–$0.6)\times10^{-5}$ at the selected amplitude and
  $(1.7$–$2.8)\times10^{-5}$ at the paper's amplitude for 1 s–1 h cutoffs.
  An independent weak-noise iSWAP calculation reproduces these values. This does not bound
  exchange-coupling noise or all flux-induced gate errors. Applying independent idle
  Gaussian dephasing formulas gives a different susceptibility, about 4% larger here.
* **The effective exponential echo rate increases with flux sensitivity.** This is
  consistent with a flux-coupled noise contribution, but an exponential envelope alone
  does not establish white noise, a particular correlation time, or a specific source.
  Q1's pure-exponential rate also mixes in its resolved Gaussian component. A joint
  exponential/Gaussian fit and noise spectroscopy are needed for a physical attribution.
* **White-noise units require a sidedness convention.** For a fitted relation
  $\Gamma_\varphi=a+c s^2$ with $s=|\partial\omega/\partial\Phi|$, a two-sided white
  PSD per Hz obeys $S_\Phi^{(2)}=2c$, while the one-sided PSD is $S_\Phi^{(1)}=4c$.
  Thus the stored $c\simeq0.0223\,{\rm s}\,\mu\Phi_0^2$ corresponds to about
  $0.21\,\mu\Phi_0/\sqrt{\rm Hz}$ two-sided, or
  $0.30\,\mu\Phi_0/\sqrt{\rm Hz}$ one-sided; the latter is about
  $0.52\,{\rm nA}/\sqrt{\rm Hz}$ at $1.7422\,{\rm mA}/\Phi_0$.
  These remain conditional, approximate levels because the quadratic fit is not clean.
* **Avoid double counting.** An independently established Markovian exponential component
  belongs in $T_\varphi$; it is not an additional error term. The stored estimate that
  69% of the idle dephasing is flux dependent is a decomposition of effective
  single-exponential rates, not a verified white-flux-noise fraction. Shared noise also
  requires care when translating the population-fit $T_\varphi$ into full gate fidelity.

## 8. Theory review: corrected derivations and remaining checks

The core section 4 derivation is correct for equal relaxation rates and independent local
exponential dephasing. In particular, the signs of the exchange terms, the factor
$2\gamma_\varphi$ in the one-excitation coherence decay, the initial-condition sine term,
and the coefficient $2/5$ for each local contribution to the two-qubit budget are correct.
The following points change the interpretation or delimit the approximation.

| Location | Finding | Correction or requirement |
|---|---|---|
| Sections 0, 2, 3 | Sum labelled `exact` is not the exact combined channel | Use `isolated-channel sum`; obtain finite gate error from the propagator |
| Sections 0, 4.1 | Exchange phase and Hilbert space were implicit | Set $\hbar=1$, truncate to two levels, and compare with the matching $U_0$ |
| Sections 3, 7 | Echo time was treated as a white-noise gate rate | Treat as conditional; separate Gaussian and exponential parts in a joint fit |
| Sections 4.5, 4.7 | Unequal-$T_1$ correction described as a relative bound | Use the exact $3\times3$ matrix; the small parameter controls absolute corrections over an appropriate window |
| Section 4.7 | Fit identifiability was stated too broadly | Pair-only identifiability applies to the reduced fit; asymmetric relaxation can be resolved in principle |
| Sections 4.7, 7 | Exchange populations omit common dephasing | A conditional population-fit budget cannot establish full average gate fidelity |
| Sections 5, 7 | Raw trace sum and noise statistics were overinterpreted | Calibrate contrasts/offsets, use binomial weights, and qualify fitted windows |
| Section 7 | Minimum pointwise upper limit called a 95% bound | Use a joint or simultaneous construction and account for poor fit quality |
| Section 7 | White-noise PSD had unspecified sidedness | $2c$ is two-sided; $4c$ is one-sided |
| Section 7 | Fixed-$g$ detuning noise called total flux error | Include coupling sensitivity and the chosen gate noise filter before making that claim |

### 8.1 Exact idle fidelity and the nonadditivity of finite errors

For one qubit with relaxation and independent exponential pure dephasing, the channel's
superoperator trace is

$$b_q=1+e^{-t/T_1^{(q)}}+2e^{-t/T_2^{(q)}}.$$

The population-transfer term is off diagonal in a population basis and does not contribute
to the trace. The two coherences each contribute $e^{-t/T_2^{(q)}}$. An independent idle
channel is a tensor product, so its trace is $b_1b_2$. For a trace-preserving channel,

$$F_{\rm ave}=\frac{\operatorname{Tr}\mathcal E+d}{d(d+1)}.$$

Taking $d=4$ gives the exact idle expression in section 2. Expansion recovers
$r_{\rm lin}=(2/5)t\sum_q(1/T_1^{(q)}+1/T_\varphi^{(q)})$.
The tensor-product terms show explicitly why isolated infidelities cannot be added exactly.

For the section 3 values and $t=492$ ns, independent calculations give:

| Quantity | Infidelity |
|---|---:|
| First-order local budget | 0.01473311 |
| Sum of exact isolated-channel contributions | 0.01466936 |
| Exact combined idle channel | 0.01457428 |
| Exact constant-rate exchange model | 0.01457750 |

The close numerical values support the first-order approximation at these rates. Their
differences do not make the isolated-channel sum an exact gate calculation.

### 8.2 Why unequal relaxation is an approximation

After removing $e^{-\gamma_1t}$, the exact system is

$$\dot{\widetilde P}=-\Delta\widetilde z,\qquad
\dot{\widetilde z}=-\Delta\widetilde P-\Omega\widetilde y,\qquad
\dot{\widetilde y}=\Omega\widetilde z-2\gamma_\varphi\widetilde y.$$

Its eigenvalues obey

$$\lambda^3+2\gamma_\varphi\lambda^2+(\Omega^2-\Delta^2)\lambda
-2\gamma_\varphi\Delta^2=0.$$

This exhibits higher-order frequency and decay shifts. Setting $\Delta=0$ makes the pair
population exactly exponential and reduces the other two equations to the damped oscillator.
For $\Delta\ne0$, the true initial slopes are
$P_d'(0)=-\gamma_1^{(d)}$, $P_a'(0)=0$, and $P'(0)=-\gamma_1^{(d)}$.
Thus the fitted harmonic-mean exponential is not an exact donor loss law at short times.
The full matrix also makes clear that equality of the two pure-dephasing rates is not required:
only their sum enters the independent-noise one-excitation block.

### 8.3 What population exchange cannot see

For a common frequency fluctuation $\xi(t)$, the noise Hamiltonian is
$\delta H=\xi(t)(n_d+n_a)$. In the one-excitation manifold $n_d+n_a=I$, so it has no
effect on exchange populations or their internal coherence. It does affect coherence between
$|00\rangle$, the one-excitation states, and $|11\rangle$.

More generally, define classical white frequency covariance by
$\langle\delta\omega_i(t)\delta\omega_j(t')\rangle=2C_{ij}\delta(t-t')$.
Then $C_{qq}$ is the individual pure-dephasing rate, while the exchange coherence decays at
$C_{dd}+C_{aa}-2C_{da}$. A population fit would identify half of this differential rate as
its effective pair $\gamma_\varphi$, not $(C_{dd}+C_{aa})/2$ unless $C_{da}=0$.
For this local frequency-noise model, the full first-order gate contribution is
$(2/5)t(C_{dd}+C_{aa})$: the cross term has zero trace contribution to the full gate average.
Additional coherence measurements are needed to connect the differential fit to that sum.

### 8.4 Quasi-static flux noise during iSWAP: the actual gate susceptibility

Assume a small constant flux offset during a square resonant gate, constant $g$, and linear
frequency shifts $\delta\omega_q=s_q\delta\Phi$ in angular-frequency units. Let
$\delta\omega_\Sigma=\delta\omega_1+\delta\omega_2$,
$\delta\omega_-=\delta\omega_1-\delta\omega_2$, and $t_g=\pi/(2g)$.
The error Hamiltonian is

$$\delta H=\delta\omega_1n_1+\delta\omega_2n_2.$$

The first-order Hermitian error generator is
$A=\int_0^{t_g}dt\,U_0^\dagger(t)\delta H U_0(t)$.
Subtracting its scalar part, the common term has eigenvalues
$(-\delta\omega_\Sigma t_g/2,0,0,+\delta\omega_\Sigma t_g/2)$.
In the one-excitation block the differential term rotates under exchange; the cosine integral
vanishes over the gate and the sine integral is $2t_g/\pi$. Its remaining generator is
$\pm(\delta\omega_-t_g/\pi)\sigma_y$.
The sign is irrelevant to the fidelity. Consequently,

$$\operatorname{Tr}A_0^2=
\frac{\delta\omega_\Sigma^2t_g^2}{2}
+\frac{2\delta\omega_-^2t_g^2}{\pi^2},\qquad A_0=A-\frac{\operatorname{Tr}A}{4}I.$$

Since $F_{\rm ave}(W,I)=(4+|\operatorname{Tr}W|^2)/20$ for a unitary error,
the leading ensemble-averaged infidelity is

$$\boxed{\langle r_\Phi\rangle=t_g^2\sigma_\Phi^2
\left[\frac{(s_1+s_2)^2}{10}+\frac{2(s_1-s_2)^2}{5\pi^2}\right]
+O(\sigma_\Phi^4).}$$

The stated remainder assumes centered Gaussian offsets, linear frequency shifts, and fixed $g$.
This differs from the independent idle Gaussian-dephasing expression
$r_{\rm idle,\Phi}=t_g^2\sigma_\Phi^2(s_1^2+s_2^2)/5+O(\sigma_\Phi^4)$.
The iSWAP partially filters differential quasi-static detuning but not the common phase.
Using the stored sensitivities gives about 4% less infidelity than the idle expression.

With $s_1/(2\pi)=-62.8$ and $s_2/(2\pi)=-107.3$ MHz per $\Phi_0$, the coefficient
$r_\Phi/\sigma_x^2$ is about $2.84\times10^4$, where $x=\Phi/\Phi_0$.
Using $\sigma_x^2=2A_x\ln[t_{\rm meas}/(2\pi t_g)]$ reproduces the stored values:

| $\sqrt{A_\Phi}$ | 1 s cutoff | 1 h cutoff |
|---|---:|---:|
| $2.29\,\mu\Phi_0$ | $3.78\times10^{-6}$ | $6.22\times10^{-6}$ |
| $4.84\,\mu\Phi_0$ | $1.69\times10^{-5}$ | $2.78\times10^{-5}$ |

Agreement with these outputs validates their fixed-$g$ frequency-offset interpretation,
conditional on the spectrum, cutoffs, and amplitude. It does not establish the true amplitude
bound or the total coupler noise. Frequencies below a recalibration bandwidth may be partly
removed; faster noise needs the full gate filter.

If the same offset changes the exchange strength by $\delta g=g_\Phi\delta\Phi$, with
$g_\Phi=\partial g/\partial\Phi$, it adds a leading contribution

$$r_{g,\Phi}=\frac25t_g^2g_\Phi^2\sigma_\Phi^2.$$

This term is orthogonal, in the trace inner product, to the common and differential generators
above, so it adds at this order in the square-gate model. It is absent from a calculation that
uses only qubit frequency derivatives. More general pulse shapes require their own sensitivity
integrals, including pump phase/amplitude and any other relevant Hamiltonian terms.

### 8.5 Statistical and implementation follow-ups

1. Relabel historical `exact` outputs as an isolated-channel sum. Optionally add the exact
   idle product formula for a direct finite-time comparison.
2. For unequal relaxation rates, implement or compare with $e^{Mt}$, especially if a fit
   aims to distinguish the qubits or characterize the first gate interval.
3. Extract $\Gamma_{\rm exp}$ from the joint Gaussian/exponential profile fit before forming
   a white-dephasing budget. A pure-exponential fit cannot isolate it on Q1.
4. For binomial samples with $N_s$ shots and predicted readout probability $m_j$, use
   $\sigma_j^2=m_j(1-m_j)/N_s$ and
   $\chi^2_{\rm red}=(N-k)^{-1}\sum_j[(m_j^{\rm obs}-m_j)/\sigma_j]^2$,
   or a binomial likelihood. Include correlations when warranted. An unweighted residual
   mean square can be compared to a shot-noise variance, but it is not this statistic.
5. Fit calibrated populations or include distinct readout contrasts explicitly. Subtract
   the offset before interpreting a log slope as a physical loss rate. The apparent
   onset step requires calibrated pump-on controls before being called population loss.
6. Reassess the pointwise $1/f$ limits with a valid joint/simultaneous construction,
   appropriate fit errors, and flux-derivative uncertainty. The profile threshold and
   treatment of a Gaussian-rate boundary must be checked in the external implementation.
7. State PSD sidedness, offset cutoffs, recalibration assumptions, and whether $g$ noise is
   included whenever quoting a flux-noise amplitude or gate error.

**Verification scope.** The attachment does not include `iswap_incoherent.py`, the device
datasets, or the flux cache. Its code cells were therefore not rerun. Independent NumPy/SciPy
calculations of the reduced ODE and full $16\times16$ Lindblad propagator verify the equations
and the section 3 gate errors. The fixed-$g$ unitary calculation verifies the quoted flux
numbers under the explicit cutoff convention. Experimental fits, their uncertainty coverage,
and external-function behavior remain unverified; their stored outputs are preserved.

**Primary references checked:** Li *et al.*, [Phys. Rev. X **14**, 041050 (2024), Appendix L](https://journals.aps.org/prx/pdf/10.1103/PhysRevX.14.041050),
and H. C. Torrey, [Phys. Rev. **76**, 1059 (1949)](https://doi.org/10.1103/PhysRev.76.1059).
All additional formulas in this review follow from the explicitly stated model.